## RAG with Azure AI search as your vector DB - Part 2 - Search

In this code sample we are going to search the vector DB in Azure AI Search we created in part 1.  
This notebook shows the basics of a RAG application, retrieval -> using GPT to get a NL (natural language) response for our query

In [1]:
import os
import json
from dotenv import load_dotenv
from tenacity import retry, wait_random_exponential, stop_after_attempt
from openai import AzureOpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SimpleField,
    SearchFieldDataType,
    SearchableField,
    SearchField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
    SemanticSearch,
    SearchIndex,
    AzureOpenAIVectorizer,
    AzureOpenAIParameters
)


from azure.identity import DefaultAzureCredential, get_bearer_token_provider
load_dotenv()
# Configure environment variables
service_endpoint = os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT")
key = os.getenv("AZURE_SEARCH_ADMIN_KEY")
index_name = os.getenv("AZURE_SEARCH_INDEX")

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_GPT4_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_GPT4_DEPLOYMENT_NAME")
AZURE_OPENAI_EMBEDDINGS_ADA_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_EMBEDDINGS_ADA_DEPLOYMENT_NAME")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
azure_openai_embedding_dimensions = 1536

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    print(ex)

In [2]:
# Configure OpenAI API
aoai_client = AzureOpenAI(
  azure_endpoint = AZURE_OPENAI_ENDPOINT, 
  api_key=AZURE_OPENAI_API_KEY,  
  api_version=AZURE_OPENAI_API_VERSION
)

In [3]:
from azure.search.documents.models import VectorizedQuery
credential = AzureKeyCredential(key)
search_client = SearchClient(endpoint=service_endpoint, index_name=index_name, credential=credential)

# Generate Document Embeddings using OpenAI Ada Model
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
# Function to generate embeddings for title and content fields, also used for query embeddings
def calc_embeddings(text):
    # model = "deployment_name"
    embeddings = aoai_client.embeddings.create(input = [text], model=AZURE_OPENAI_EMBEDDINGS_ADA_DEPLOYMENT_NAME).data[0].embedding
    return embeddings

def do_search(query):
    fields = "embedding"
    embedding = calc_embeddings(query)
    vector_query = VectorizedQuery(vector=embedding, k_nearest_neighbors=3, fields=fields)
  
    results = search_client.search(  
        search_text=None,  
        vector_queries= [vector_query],
        select=["content"],
    )  
    answer = ''
    for result in results:  
        print(f"Score: {result['@search.score']}")  
        print(f"Content: {result['content']}")  
        answer = answer + result['content']
    return answer

In [4]:
question = "What's Microsoft Fabric?"
answers = do_search(question)
print(answers)

Score: 0.90817213
Content: ） Important
Microsoft Fabric is currently in PREVIEW. This information relates to a prerelease
product that may be substantially modified before it's released. Microsoft makes no
warranties, expressed or implied, with respect to the information provided here.
Score: 0.90277976
Content: ）  Important
Microsoft Fabric is currently in PREVIEW. This information relates to a prerelease
product that may be substantially modified before it's released. Microsoft makes no
warranties, expressed or implied, with respect to the information provided here.
Next steps
Score: 0.8925502
Content: Lakehouse end-to-end scenario:
overview and architecture
Article• 05/23/2023
Microsoft Fabric is an all-in-one analytics solution for enterprises that covers everything
from data movement to data science, real-time analytics, and business intelligence. It
offers a comprehensive suite of services, including data lake, data engineering, and data
integration, all in one place. For more in

In [5]:
def call_openAI(text):
    response = aoai_client.chat.completions.create(
        model=AZURE_OPENAI_GPT4_DEPLOYMENT_NAME,
        messages = text,
        temperature=0.7,
        max_tokens=800,
        top_p=0.95,
        frequency_penalty=0,
        presence_penalty=0,
        stop=None
    )

    return response.choices[0].message.content

In [6]:
# This prompt provides instructions to the model. 
# The prompt includes the query and the source, which are specified further down in the code.
grounded_prompt="""
You are a friendly assistant answering users questions.
Answer the query using only the answers provided below in a friendly and concise bulleted manner.
Answer ONLY with the facts listed in the list of answers below.
If there isn't enough information below, say you don't know.
Do not generate answers that don't use the answers below.
Query: {question}
Sources:\n{answers}
"""
# prepare prompt
messages=[
    {
        "role": "user",
        "content": grounded_prompt.format(question=question, answers=answers)
    }
]
  
result = call_openAI(messages)
display(result)

'- Microsoft Fabric is currently in PREVIEW.\n- It is an all-in-one analytics solution for enterprises.\n- It covers data movement, data science, real-time analytics, and business intelligence.\n- Includes comprehensive services like data lake, data engineering, and data integration all in one place.\n- The information provided about Microsoft Fabric may be substantially modified before its official release.'